# Map Plotting

This notebook contains various map plots. It produces Figures 1, 3, 6 and 9, and serves as the source for data used throughout the paper.

It requires the following data sources:
- `enriched_cf_sept_all.csv`: Cloudflare data before client-server filtering
- `enriched_cf_sept_after.csv`: Cloudflare data after client-server filtering
- `enriched_ndt7_sept_all.csv`: NDT7 data before client-server filtering
- `enriched_ndt7_sept_after.csv`: NDT7 data after client-server filtering
- `latency_results.csv`: Prediction results for 24 November 2025
- `throughput_results.csv`: Prediction results for 24 November 2025

The pops and ground stations data will be automatically downloaded.

In [8]:
import sys
import os
import pandas as pd
import importlib
from pathlib import Path
import os
from typing import Optional, Set, Tuple



sys.path.append(os.path.abspath('..'))

import utils
importlib.reload(utils)
from utils import (
    Aggregation,
    filter_filtration_dfs_by_min_measurements,
    get_download_and_upload_df,
)

import map_plotting
importlib.reload(map_plotting)
from map_plotting import (
    plot_feature,
    plot_three_datasets,
    plot_filtering_percentage,
    plot_starlink_infrastructure_map,
    OrbitToParams
)

import get_pop_and_gs
importlib.reload(get_pop_and_gs)
from get_pop_and_gs import save_pop_and_gs

import get_starlink_availability
importlib.reload(get_starlink_availability)
from get_starlink_availability import save_countries_to_file

import orbit_utils
importlib.reload(orbit_utils)
from orbit_utils import get_coords

In [9]:
output_dir = Path("./output")
os.makedirs(output_dir, exist_ok=True)
data_dir = Path("./data")

cf_all = pd.read_csv(data_dir / 'enriched_cf_sept_all.csv')
cf_after = pd.read_csv(data_dir / 'enriched_cf_sept_after.csv')
cf_filtered = cf_all[~cf_all['uuid'].isin(cf_after['uuid'])]
assert len(cf_filtered) + len(cf_after) == len(cf_all), "Filtered and after datasets should have the same number of entries."

ndt_all = pd.read_csv(data_dir / 'enriched_ndt7_sept_all.csv')
ndt_after = pd.read_csv(data_dir / 'enriched_ndt7_sept_after.csv')
ndt_filtered = ndt_all[~ndt_all['uuid'].isin(ndt_after['uuid'])]
assert len(ndt_filtered) + len(ndt_after) == len(ndt_all), "Filtered and after datasets should have the same number of entries."

latency_res = pd.read_csv(data_dir / 'latency_results.csv')
throughput_res = pd.read_csv(data_dir / 'throughput_results.csv')


starlink_countries_path = data_dir / 'starlink_countries.csv'
if not starlink_countries_path.exists():
    print("Starlink availability data file is missing.")
    save_countries_to_file()
starlink_countries: list[str] = pd.read_csv(starlink_countries_path)['country_code'].tolist()

C:\Users\vladg\AppData\Local\Temp\ipykernel_18864\3728194511.py:11: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  ndt_after = pd.read_csv(data_dir / 'enriched_ndt7_sept_after.csv')


In [10]:
def process_and_plot_filtration_for_dataset(
        dataset_name: str,
        df_all: pd.DataFrame,
        df_filtered: pd.DataFrame,
        df_after: pd.DataFrame,
        min_measurements: int,
        allowed_countries: Optional[list[str]]=None) -> None:
    print(f"Average percentage removed in {dataset_name}: {100 * (len(df_filtered) / len(df_all)):.2f}%")

    df_all, df_filtered, df_after = filter_filtration_dfs_by_min_measurements(
        df_all, df_filtered, df_after, min_measurements, allowed_countries=allowed_countries
    )
    
    download_all, upload_all = get_download_and_upload_df(df_all)
    download_filtered, upload_filtered = get_download_and_upload_df(df_filtered)
    download_after, upload_after = get_download_and_upload_df(df_after)
    
    print(f"\n{'='*60}")
    print(f"{dataset_name} Plots")
    print('='*60)

    print(f"\nDownload Latency Median Before")
    fig = plot_feature(
        download_all,
        feature='download_latency_ms',
        aggregation=Aggregation.MEDIAN,
        tick_step=50,
    )
    fig.show()

    print(f"\nDownload Throughput Median Before")
    fig = plot_feature(
        download_all,
        feature='download_throughput_mbps',
        aggregation=Aggregation.MEDIAN,
        tick_step=100,
    )
    fig.show()

    print(f"\n{dataset_name} - Download Latency Median Comparison")
    fig = plot_three_datasets(
        download_all, download_filtered, download_after,
        feature='download_latency_ms',
        aggregation=Aggregation.MEDIAN
    )

    fig.show()
    
    print(f"\n{dataset_name} - Download Throughput Median Comparison")
    fig = plot_three_datasets(
        download_all, download_filtered, download_after,
        feature='download_throughput_mbps',
        aggregation=Aggregation.MEDIAN,
    )
    fig.show()
    
    print(f"\n{dataset_name} - Client-Server Distance Mean Comparison")
    fig = plot_three_datasets(
        df_all, df_filtered, df_after,
        feature='client_server_distance_km',
        aggregation=Aggregation.MEAN,
    )
    fig.show()

    print(f"\n{dataset_name} - Filtering Percentage")
    fig = plot_filtering_percentage(df_all, df_filtered)
    fig.show()

In [11]:
def plot_cf_and_ndt(ndt_df: pd.DataFrame, cf_df: pd.DataFrame) -> None:
    ndt_all_df, _ = get_download_and_upload_df(ndt_df)
    cf_all_df = cf_df.copy()
    df = pd.concat([ndt_all_df, cf_all_df], ignore_index=True)

    df, _, _ = filter_filtration_dfs_by_min_measurements(
        df, df, df, 1, allowed_countries=starlink_countries
    )

    print(f"\nDownload Latency Median Before")
    fig = plot_feature(
        df,
        feature='download_latency_ms',
        aggregation=Aggregation.MEDIAN
    )
    fig.show()
    fig.write_image(output_dir / "median_download_latency.pdf")


    print(f"\nDownload Throughput Median Before")
    fig = plot_feature(
        df,
        feature='download_throughput_mbps',
        aggregation=Aggregation.MEDIAN
    )
    fig.show()
    fig.write_image(output_dir / "median_download_throughput.pdf")

In [12]:
def plot_starlink_infrastructure( islands_manual_plot: Set[Tuple[float, float]], orbit_to_params: OrbitToParams) -> None:
    pop_path = data_dir / 'pops.csv'
    gs_path = data_dir / 'ground_stations.csv'

    if not pop_path.exists() or not gs_path.exists():
        print("PoP or GS data files are missing.")
        save_pop_and_gs()

    pops = pd.read_csv(pop_path)
    gs = pd.read_csv(gs_path)

    fig = plot_starlink_infrastructure_map(islands_manual_plot, pops, gs, orbit_to_params, starlink_countries)
    fig.show()
    fig.write_image(output_dir / "starlink_infra.pdf")

In [13]:
def plot_evaluation_results(width = 740, height = 300, use_common_countries = True) -> None:
    if use_common_countries:
        latency_countries = latency_res['country']
        throughput_countries = throughput_res['country']
        common_countries = set(latency_countries) & set(throughput_countries)

        latency_df = latency_res[latency_res['country'].isin(common_countries)].copy()
        throughput_df = throughput_res[throughput_res['country'].isin(common_countries)].copy()
    else:
        latency_df = latency_res.copy()
        throughput_df = latency_res.copy()
    
    for feature in ['mae', 'rmse']:
        print(f'Plotting {feature} for latency')
        fig = plot_feature(latency_df, feature, unit = 'ms', country_col='country', aggregation=Aggregation.NONE, tick_step=10, width=width, height=height)
        fig.show()
        fig.write_image(output_dir / f"latency_{feature}_results.pdf")
        
        print(f'Plotting {feature} for throughput')
        fig = plot_feature(throughput_df, feature, unit = 'Mbps', country_col='country', aggregation=Aggregation.NONE, tick_step=20, width=width, height=height)
        fig.show()
        fig.write_image(output_dir / f"throughput_{feature}_results.pdf")

In [ ]:
begin_time = 1767998263
orbit_to_params: OrbitToParams = {
        43.0: {
            'coords': get_coords(  43, 238, 540, 100, begin_time, portion=1),
            'num_sats': 2900,
            'linestyle': 'dot'
        },
        53.0: {
            'coords': get_coords(  53, 238, 550, 100, begin_time, portion=1),
            'num_sats': 3800,
            'linestyle': 'dash'
        },
        70.0: {
            'coords': get_coords(  70, 238, 570, 100, begin_time, portion=1),
            'num_sats': 500,
            'linestyle': 'dashdot'
        },
        97.6: {
            'coords': get_coords(97.6,  70, 560, 100, begin_time, portion=1),
            'num_sats': 200,
            'linestyle': 'longdashdot'
        },
    }

# Very small islands need to be manually plotted
manual_islands = {
    (13.06276, -59.53758), (12.09769, -68.90811), (-0.7451734, -90.31547), (53.86028, -166.5049),
    (-8.526697, 179.1926), (1.332318, 173.0041), (-0.52913, 166.9177), (5.32539, 163.0126), (13.61636, 144.8587)
}

# This generates Figure 1
plot_starlink_infrastructure(manual_islands, orbit_to_params)

In [ ]:
# THis generates Figure 3
plot_cf_and_ndt(ndt_all, cf_all)

Countries after allowed list filter: 122
Countries removed: 25

Records before: df_all=1448869, df_filtered=1448869, df_after=1448869
Records after: df_all=1392662, df_filtered=1392662, df_after=1392662

Download Latency Median Before



Download Throughput Median Before


In [ ]:
# THis generates Figure 6 and 9
plot_evaluation_results()

Plotting mae for latency


Plotting mae for throughput


Plotting rmse for latency


Plotting rmse for throughput


## Client-Server Filtering

These plots are not used directly in the paper, but the data visualized in them is mentioned in the discussions.

### Cloudflare Dataset

In [17]:
process_and_plot_filtration_for_dataset(
    "Cloudflare",
    cf_all,
    cf_filtered,
    cf_after,
    min_measurements=50,
    allowed_countries=starlink_countries
)

Average percentage removed in Cloudflare: 9.15%
Countries after allowed list filter: 38
Countries removed: 82

Records before: df_all=18070, df_filtered=1653, df_after=16417
Records after: df_all=15798, df_filtered=1357, df_after=14441

Cloudflare Plots

Download Latency Median Before



Download Throughput Median Before



Cloudflare - Download Latency Median Comparison



Cloudflare - Download Throughput Median Comparison



Cloudflare - Client-Server Distance Mean Comparison



Cloudflare - Filtering Percentage


### NDT7 Dataset

In [18]:
process_and_plot_filtration_for_dataset(
    "NDT7",
    ndt_all,
    ndt_filtered,
    ndt_after,
    min_measurements=50,
    allowed_countries=starlink_countries
)

Average percentage removed in NDT7: 5.82%
Countries after allowed list filter: 96
Countries removed: 50

Records before: df_all=2485385, df_filtered=144606, df_after=2340779
Records after: df_all=2387186, df_filtered=137388, df_after=2249798

NDT7 Plots

Download Latency Median Before



Download Throughput Median Before



NDT7 - Download Latency Median Comparison



NDT7 - Download Throughput Median Comparison



NDT7 - Client-Server Distance Mean Comparison



NDT7 - Filtering Percentage
